# Import Dependencies

In [1]:
import pandas as pd
from os import path, getcwd

# Create Variables

In [2]:
dataset_dir = '..\\data\\dataset'
train_path = path.join(dataset_dir, 'raw', 'train.parquet')
test_path = path.join(dataset_dir, 'raw', 'test.parquet')
val_path = path.join(dataset_dir, 'raw', 'val.parquet')
raw_path = path.join(dataset_dir, 'raw', 'raw.parquet')

# Read Data

In [3]:
df_train = pd.read_parquet(train_path)
print("Train data shape:", df_train.shape)
df_test = pd.read_parquet(test_path)
print("Test data shape:", df_test.shape)
df_val = pd.read_parquet(val_path)
print("Validation data shape:", df_val.shape)
df_raw = pd.read_parquet(raw_path)
print("Raw data shape:", df_raw.shape)
print(df_raw.columns)

Train data shape: (166251, 3)
Test data shape: (20781, 3)
Validation data shape: (20782, 3)
Raw data shape: (211222, 37)
Index(['text', 'id', 'author', 'subreddit', 'link_id', 'parent_id',
       'created_utc', 'rater_id', 'example_very_unclear', 'admiration',
       'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion',
       'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust',
       'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy',
       'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief',
       'remorse', 'sadness', 'surprise', 'neutral'],
      dtype='object')


# Concatinate train, test and validation set for preprocessing

In [4]:
df_concat = pd.concat([df_train, df_test, df_val], ignore_index=True).drop_duplicates(subset=['text']).reset_index(drop=True)
print("Concatenated data shape:", df_concat.shape)
df_concat.head()

Concatenated data shape: (57941, 3)


,text,labels,id
0,Ik mis de oude [NAME],[25],ee618dn
1,"ik heb [NAAM] een beetje een afstandsschot, ma...",[27],efcz2dz
2,Kan PG gewoon chillen,[27],eczgifx
3,Zijn polsen zijn te smal ☹️,[25],ef3y2oy
4,Daenerys heeft [NAME] een valyriaans stalen zw...,"[0, 7]",eeiuscs


# Merge with raw dataset to get encodings of fine-grained emotions 

In [5]:
df_combined = pd.merge(df_concat, df_raw, on='id', how='inner').drop_duplicates(subset=['text_x'])
print("Combined data shape:", df_combined.shape)
df_combined.head()

Combined data shape: (57941, 39)


,text_x,labels,id,text_y,author,subreddit,link_id,parent_id,created_utc,rater_id,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,Ik mis de oude [NAME],[25],ee618dn,I miss old [NAME],Maurice_Acuna22,devils,t3_ages1s,t1_ee615zr,1.547606e+09,70,...,1,0,0,0,0,0,0,0,0,0
5,"ik heb [NAAM] een beetje een afstandsschot, ma...",[27],efcz2dz,i got [NAME] a bit of a long shot but i see why,Eggsworth,Persona5,t3_al6r2w,t3_al6r2w,1.548859e+09,49,...,0,0,0,0,0,0,0,0,0,1
8,Kan PG gewoon chillen,[27],eczgifx,Can PG just chill,niggasbelike____,Mavericks,t3_ab5f7o,t3_ab5f7o,1.546306e+09,15,...,0,0,0,0,0,0,0,0,0,0
13,Zijn polsen zijn te smal ☹️,[25],ef3y2oy,His wrists are too small ☹️,OigoMiEggo,TopMindsOfReddit,t3_ak9hcz,t1_ef3fmsx,1.548607e+09,7,...,0,0,0,0,0,0,0,1,0,0
18,Daenerys heeft [NAME] een valyriaans stalen zw...,"[0, 7]",eeiuscs,[NAME] is promised a valyrian steel sword by d...,k0w4l5kii,asoiaf,t3_ahlib4,t3_ahlib4,1.547985e+09,23,...,0,0,0,0,0,0,0,0,0,0


# Drop unnecessary columns & rename

In [6]:
df_combined = df_combined.drop(
    columns=[
        'id', 
        'author',
        'labels',
        'text_y',
        'subreddit', 
        'link_id', 
        'parent_id',
        'created_utc', 
        'rater_id', 
        'example_very_unclear'
    ]
).rename(columns={'text_x': 'Sentence'})
df_combined.head()

,Sentence,admiration,amusement,anger,annoyance,approval,caring,confusion,curiosity,desire,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,Ik mis de oude [NAME],0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
5,"ik heb [NAAM] een beetje een afstandsschot, ma...",0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
8,Kan PG gewoon chillen,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
13,Zijn polsen zijn te smal ☹️,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
18,Daenerys heeft [NAME] een valyriaans stalen zw...,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


# Unhot encode and map fine-grained emotions to basic emotions

In [7]:
# Emotion map
emotion_map = {
    'admiration': 'happiness',
    'amusement': 'happiness',
    'anger': 'anger',
    'annoyance': 'anger',
    'approval': 'happiness',
    'caring': 'happiness',
    'confusion': 'surprise',
    'curiosity': 'surprise',
    'desire': 'happiness',
    'disappointment': 'sadness',
    'disapproval': 'sadness',
    'disgust': 'disgust',
    'embarrassment': 'sadness',
    'excitement': 'happiness',
    'fear': 'fear',
    'gratitude': 'happiness',
    'grief': 'sadness',
    'joy': 'happiness',
    'love': 'happiness',
    'nervousness': 'fear',
    'optimism': 'happiness',
    'pride': 'happiness',
    'realization': 'surprise',
    'relief': 'happiness',
    'remorse': 'sadness',
    'sadness': 'sadness',
    'surprise': 'surprise',
    'neutral': 'neutral'
}

# emotion columns 
emotion_cols = list(emotion_map.keys())

# melt to long format
df_final = df_combined.melt(
    id_vars=[c for c in df_combined.columns if c not in emotion_cols], 
    value_vars=emotion_cols, 
    var_name="emotion", 
    value_name="value"
)

# keep only active emotions (1 or >0 if continuous)
df_final = df_final[df_final["value"] > 0].copy()
df_final["Emotion"] = df_final["emotion"].map(emotion_map)

# if you want one row per original sample with category instead of one-hot
df_final = df_final.drop(columns=["emotion", "value"]).reset_index(drop=True)

unique_emotions = df_final['Emotion'].unique()
emotion_mapping = {emotion: i for i, emotion in enumerate(unique_emotions)}
df_final['Emotion_encoded'] = df_final['Emotion'].map(emotion_mapping)
df_final = df_final.drop_duplicates(subset=['Sentence']).reset_index(drop=True)
df_final.head()

,Sentence,Emotion,Emotion_encoded
0,Bedankt! Duidelijke rekwisieten voor hem omdat...,happiness,0
1,Goed werk!,happiness,0
2,ngl zijn totale diefstal van de lwymmd-teksten...,happiness,0
3,Wat een frisse en unieke kritiek!,happiness,0
4,alle lof zij de donkere moeder!,happiness,0


# Check output

In [8]:
sampled_sentences = df_final.groupby('Emotion').apply(lambda x: x.sample(n=10, replace=True, random_state=1)).reset_index(drop=True)

# Print the emotion to integer mapping.
print("--- Emotie naar integer mapping ---")
for emotion, encoded_value in emotion_mapping.items():
    print(f"'{emotion}': {encoded_value}")
print("-" * 35)

# Print the 10 random sentences for each emotion.
for emotion in sampled_sentences['Emotion'].unique():
    print(f"\nEmotie: {emotion}")
    emotion_df = sampled_sentences[sampled_sentences['Emotion'] == emotion]
    for index, row in emotion_df.iterrows():
        print(f"- {row['Sentence']}")

--- Emotie naar integer mapping ---
'happiness': 0
'anger': 1
'surprise': 2
'sadness': 3
'disgust': 4
'fear': 5
'neutral': 6
-----------------------------------

Emotie: anger
- Hier ook. Ik had veel zwelling maar geen abnormale pijn.
- Feit is dat de lucht oranje is. En als je denkt dat het blauw is, ben je dom
- Te lelijk voor liefde/romantiek/seks/dating
- Ik wil een hele rol kwartjes van die kont stuiteren, heilige shit.
- Who da fuck draagt ​​zwembroek naar een wedstrijd
- Allereerst gecondoleerd met uw verlies. Houd er rekening mee dat begrafenissen voor de mensen zijn die nog in leven zijn, niet voor de persoon die net is overleden.
- Dat komt omdat je bekrompen bent. Maar bleh. Ik heb je respect niet nodig. ;)
- Man, deze onderzeeër is zo elitair en verwaand geworden.
- Het is voetbal. Hij deed hoe dan ook alsof hij geblesseerd was.
- Je hebt hem net verdedigd met een hoop dubbele onzin, dus ik ben het er niet mee eens. Je bent een verdomde idioot.

Emotie: disgust
- 2 is uitge

C:\Users\nickb\AppData\Local\Temp\ipykernel_17792\2271713206.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_sentences = df_final.groupby('Emotion').apply(lambda x: x.sample(n=10, replace=True, random_state=1)).reset_index(drop=True)


# Save Dataframe

In [9]:
print(getcwd())
df_final.to_csv(path.join(dataset_dir, 'processed', 'go_emotion_dutch.csv'), index=False)

c:\Users\nickb\OneDrive\Documenten\GitHub\fae2-nlpr-group-group-4-1\nlp_cia\src
